In [ ]:
!pip install pdfplumber

### Tahap 1: Membangun Case Base (Inisialisasi)

In [ ]:
import os
from google.colab import files

# Mendefinisikan struktur direktori sesuai variabel yang Anda inginkan
DATA_DIRS = ['data/raw/', 'data/cleaned/', 'input_pdfs']

def setup_environment(dirs):
    for d in dirs:
        # Perbaikan: menggunakan os.path.exists
        if not os.path.path.exists(d) if hasattr(os, 'path') else not os.path.exists(d):
            # Standar yang benar adalah os.path.exists
            if not os.path.exists(d) and not os.path.isdir(d):
                pass
    # Menulis ulang fungsi dengan sintaks yang benar
    for d in dirs:
        if not os.path.exists(d):
            os.makedirs(d)
            print(f"Direktori created: {d}")
        else:
            print(f"Direktori sudah ada: {d}")

# Memperbaiki fungsi setup secara eksplisit
def setup_environment_fixed(dirs):
    for d in dirs:
        if not os.path.exists(d):
            os.makedirs(d)
            print(f"Direktori created: {d}")
        else:
            print(f"Direktori sudah ada: {d}")

setup_environment_fixed(DATA_DIRS)

# Variabel akses cepat
DATA_RAW = 'data/raw/'
DATA_CLEANED = 'data/cleaned/'
PDF_FOLDER = 'input_pdfs'

Direktori sudah ada: data/raw/
Direktori sudah ada: data/cleaned/
Direktori sudah ada: input_pdfs


In [ ]:
import os
import re
import pdfplumber
import shutil
from google.colab import files

def clear_folder(folder_path):
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Gagal menghapus {file_path}. Alasan: {e}')

def upload_pdf_files(target_folder):
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
    print("Pilih file PDF Putusan MA untuk diunggah:")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        path = os.path.join(target_folder, filename)
        with open(path, 'wb') as f:
            f.write(content)
        print(f"[TERUNGGAH] {filename}")

def clean_legal_text(text):
    text = text.lower()
    patterns = [
        r'halaman \d+ dari \d+',
        r'direktori putusan mahkamah agung republik indonesia',
        r'putusan\.mahkamahagung\.go\.id',
        r'disclamer : .*$',
        r'kepaniteraan mahkamah agung republik indonesia',
        r'mahkamah agung republik indonesia',
    ]
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'[^a-z0-9\s/.,]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def process_cases(source_folder, cleaned_folder, raw_folder, clear_first=True):
    if clear_first:
        clear_folder(cleaned_folder)
        clear_folder(raw_folder)

    if not os.path.exists(cleaned_folder): os.makedirs(cleaned_folder)
    if not os.path.exists(raw_folder): os.makedirs(raw_folder)

    pdf_files = [f for f in os.listdir(source_folder) if f.endswith('.pdf')]

    for i, filename in enumerate(pdf_files, start=1):
        new_name_txt = f"case_{i:03d}.txt"
        new_name_pdf = f"case_{i:03d}.pdf"
        src_path = os.path.join(source_folder, filename)

        shutil.copy(src_path, os.path.join(raw_folder, new_name_pdf))

        try:
            all_text = ""
            with pdfplumber.open(src_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text: all_text += page_text + "\n"

            cleaned_content = clean_legal_text(all_text)

            with open(os.path.join(cleaned_folder, new_name_txt), 'w', encoding='utf-8') as f:
                f.write(cleaned_content)

            print(f"[PROSES BERHASIL] {filename}: Terkonversi dan dibersihkan.")

        except Exception as e:
            print(f"[GAGAL] {filename}: {str(e)}")

In [ ]:
import os

# 1. Bersihkan folder input agar tidak ada sisa file dari sesi sebelumnya
clear_folder(PDF_FOLDER)

# 2. Unggah file baru
upload_pdf_files(PDF_FOLDER)

# 3. Bersihkan folder tujuan (raw & cleaned) untuk sinkronisasi total
clear_folder(DATA_RAW)
clear_folder(DATA_CLEANED)

# 4. Proses hanya file yang baru diunggah
process_cases(PDF_FOLDER, DATA_CLEANED, DATA_RAW, clear_first=False)

# 5. Rapikan direktori /content/
for f in os.listdir('/content/'):
    if f.endswith('.pdf') and ('(' in f or 'putusan' in f.lower()):
        try: os.remove(os.path.join('/content/', f))
        except: pass

print("\n--- Sinkronisasi Selesai ---")
print(f"File PDF di data/raw: {len(os.listdir(DATA_RAW))}")
print(f"File TXT di data/cleaned: {len([f for f in os.listdir(DATA_CLEANED) if f.endswith('.txt')])}")

Pilih file PDF Putusan MA untuk diunggah:


Saving putusan_303_pid.sus_2025_pn_yyk_20260626133419.pdf to putusan_303_pid.sus_2025_pn_yyk_20260626133419.pdf
Saving putusan_304_pid.sus_2025_pn_yyk_20260626133410.pdf to putusan_304_pid.sus_2025_pn_yyk_20260626133410.pdf
Saving putusan_316_pid.sus_2025_pn_yyk_20260626133503.pdf to putusan_316_pid.sus_2025_pn_yyk_20260626133503.pdf
Saving putusan_324_pid.sus_2025_pn_yyk_20260626133630.pdf to putusan_324_pid.sus_2025_pn_yyk_20260626133630.pdf
Saving putusan_332_pid.sus_2025_pn_yyk_20260626133625.pdf to putusan_332_pid.sus_2025_pn_yyk_20260626133625.pdf
Saving putusan_336_pid.sus_2025_pn_yyk_20260626133341.pdf to putusan_336_pid.sus_2025_pn_yyk_20260626133341.pdf
Saving putusan_340_pid.sus_2025_pn_yyk_20260626133601.pdf to putusan_340_pid.sus_2025_pn_yyk_20260626133601.pdf
Saving putusan_346_pid.sus_2025_pn_yyk_20260626133317.pdf to putusan_346_pid.sus_2025_pn_yyk_20260626133317.pdf
Saving putusan_353_pid.sus_2025_pn_yyk_20260626133329.pdf to putusan_353_pid.sus_2025_pn_yyk_20260626133

In [ ]:
import os

def validate_cleaning(cleaned_folder, log_path='cleaning.log'):
    if not os.path.exists('logs'): os.makedirs('logs')
    log_file = os.path.join('logs', log_path)

    print(f"--- Memulai Validasi di {cleaned_folder} ---")
    with open(log_file, 'w') as log:
        for filename in sorted(os.listdir(cleaned_folder)):
            if filename.endswith('.txt'):
                path = os.path.join(cleaned_folder, filename)
                with open(path, 'r', encoding='utf-8') as f:
                    content = f.read()

                # Kriteria 1: Periksa keutuhan teks (contoh: minimal 1000 karakter atau cek keyword)
                char_count = len(content)
                has_essential = 'menimbang' in content and 'mengadili' in content
                status = "VALID" if (char_count > 1000 and has_essential) else "PERLU CEK MANUAL"

                log_msg = f"[{status}] {filename}: {char_count} karakter, Keywords found: {has_essential}\n"
                log.write(log_msg)
                print(log_msg.strip())

# Jalankan validasi setelah proses sinkronisasi selesai
validate_cleaning(DATA_CLEANED)

--- Memulai Validasi di data/cleaned/ ---
[VALID] case_001.txt: 145258 karakter, Keywords found: True
[VALID] case_002.txt: 52173 karakter, Keywords found: True
[VALID] case_003.txt: 109990 karakter, Keywords found: True
[VALID] case_004.txt: 57071 karakter, Keywords found: True
[VALID] case_005.txt: 119993 karakter, Keywords found: True
[VALID] case_006.txt: 96634 karakter, Keywords found: True
[VALID] case_007.txt: 93383 karakter, Keywords found: True
[VALID] case_008.txt: 54600 karakter, Keywords found: True
[VALID] case_009.txt: 101918 karakter, Keywords found: True
[VALID] case_010.txt: 205746 karakter, Keywords found: True
[VALID] case_011.txt: 67648 karakter, Keywords found: True
[VALID] case_012.txt: 104970 karakter, Keywords found: True
[VALID] case_013.txt: 115819 karakter, Keywords found: True
[VALID] case_014.txt: 87225 karakter, Keywords found: True
[VALID] case_015.txt: 74799 karakter, Keywords found: True


In [ ]:
# Menampilkan isi cleaning log untuk verifikasi output Tahap 1
log_path = 'logs/cleaning.log'

if os.path.exists(log_path):
    print(f"--- Isi File: {log_path} ---\n")
    with open(log_path, 'r') as f:
        print(f.read())
else:
    print("File log belum ditemukan. Pastikan sel validasi sudah dijalankan.")

--- Isi File: logs/cleaning.log ---

[VALID] case_001.txt: 145258 karakter, Keywords found: True
[VALID] case_002.txt: 52173 karakter, Keywords found: True
[VALID] case_003.txt: 109990 karakter, Keywords found: True
[VALID] case_004.txt: 57071 karakter, Keywords found: True
[VALID] case_005.txt: 119993 karakter, Keywords found: True
[VALID] case_006.txt: 96634 karakter, Keywords found: True
[VALID] case_007.txt: 93383 karakter, Keywords found: True
[VALID] case_008.txt: 54600 karakter, Keywords found: True
[VALID] case_009.txt: 101918 karakter, Keywords found: True
[VALID] case_010.txt: 205746 karakter, Keywords found: True
[VALID] case_011.txt: 67648 karakter, Keywords found: True
[VALID] case_012.txt: 104970 karakter, Keywords found: True
[VALID] case_013.txt: 115819 karakter, Keywords found: True
[VALID] case_014.txt: 87225 karakter, Keywords found: True
[VALID] case_015.txt: 74799 karakter, Keywords found: True



In [ ]:
import os

def check_text_integrity(cleaned_folder):
    print(f"=== LAPORAN KEUTUHAN TEKS (OUTPUT TAHAP 1) ===\n")

    for filename in sorted(os.listdir(cleaned_folder)):
        if filename.endswith('.txt'):
            path = os.path.join(cleaned_folder, filename)
            with open(path, 'r', encoding='utf-8') as f:
                content = f.read()

            char_count = len(content)
            # Estimasi sederhana: dokumen hukum rata-rata > 5000 karakter
            # Kita hitung persentase terhadap ambang batas kelayakan
            integrity_score = min(100, (char_count / 5000) * 100)

            has_menimbang = 'menimbang' in content.lower()
            has_mengadili = 'mengadili' in content.lower()

            print(f"File: {filename}")
            print(f"- Estimasi Keutuhan: {integrity_score:.2f}%")
            print(f"- Struktur Menimbang: {'[OK]' if has_menimbang else '[MISSING]'}")
            print(f"- Struktur Mengadili: {'[OK]' if has_mengadili else '[MISSING]'}")
            print(f"- Status Akhir: {'MEMENUHI SYARAT (>= 80%)' if integrity_score >= 80 and has_menimbang and has_mengadili else 'PERLU REVISI'}")
            print("-" * 40)

check_text_integrity(DATA_CLEANED)

=== LAPORAN KEUTUHAN TEKS (OUTPUT TAHAP 1) ===

File: case_001.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_002.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_003.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_004.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_005.txt
- Estimasi Keutuhan: 100.00%
- Struktur Menimbang: [OK]
- Struktur Mengadili: [OK]
- Status Akhir: MEMENUHI SYARAT (>= 80%)
----------------------------------------
File: case_006.txt
- Estimasi Ke

### Tahap 2: Case Representation (Strukturisasi Data)

In [ ]:
import json
import glob
import re
import os

def represent_case(raw_text):
    """
    Implementasi Tahap 2: Mengekstrak fitur kunci (Fakta & Amar)
    berdasarkan struktur standar Putusan MA.
    """
    nomor_match = re.search(r"putusan nomor (\d+/[^/]+/[^/]+/[^/]+)", raw_text, re.I)
    nomor_putusan = nomor_match.group(0) if nomor_match else "Tidak Diketahui"

    fakta_start = raw_text.find("menimbang")
    amar_start = raw_text.find("mengadili")

    fakta_hukum = ""
    amar_putusan = ""

    if fakta_start != -1 and amar_start != -1:
        fakta_hukum = raw_text[fakta_start:amar_start].strip()
        amar_putusan = raw_text[amar_start:].strip()
    elif fakta_start != -1:
        fakta_hukum = raw_text[fakta_start:fakta_start+3000].strip()
    else:
        fakta_hukum = raw_text[:2000]

    case_data = {
        "nomor_putusan": nomor_putusan,
        "fakta_hukum": fakta_hukum,
        "amar_putusan": amar_putusan,
        "kategori": "Pidana"
    }

    return case_data

def save_to_structured_base(source_folder, output_file):
    case_base = []
    # Perbaikan: Mencari file .txt di folder cleaned
    files_found = glob.glob(os.path.join(source_folder, "case_*.txt"))

    for filepath in files_found:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
            case_representation = represent_case(text)
            case_base.append(case_representation)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(case_base, f, indent=4)

    print(f"Case Base diperbarui di {output_file}. Total kasus terstruktur: {len(case_base)}")

# Eksekusi Tahap 2: Membaca dari DATA_CLEANED
structured_file = os.path.join(DATA_CLEANED, 'case_base.json')
save_to_structured_base(DATA_CLEANED, structured_file)

Case Base diperbarui di data/cleaned/case_base.json. Total kasus terstruktur: 15


In [ ]:
import re
import json
import glob
import os

def represent_case_v2(raw_text):
    # 1. Nomor Putusan
    nomor_match = re.search(r"putusan nomor (\d+/[^/]+/[^/]+/[^/]+)", raw_text, re.I)
    nomor_putusan = nomor_match.group(1) if nomor_match else "Tidak Diketahui"

    # 2. EKSTRAKSI METADATA
    tanggal_match = re.search(r"tanggal\s+(\d+\s+[a-z]+\s+\d{4})", raw_text, re.I)
    tanggal = tanggal_match.group(1) if tanggal_match else "Tidak Terdeteksi"

    pihak_match = re.search(r"nama lengkap\s+([^;]+)\s+tempat lahir", raw_text, re.I)
    if not pihak_match:
        pihak_match = re.search(r"terdakwa\s+([^;\n]+)", raw_text, re.I)
    pihak = pihak_match.group(1).strip() if pihak_match else "Tidak Terdeteksi"

    pasal_match = re.search(r"pasal\s+(\d+[^,.]*)", raw_text, re.I)
    pasal = pasal_match.group(0).strip() if pasal_match else "Tidak Terdeteksi"

    # 3. Pemisahan Fakta & Amar (Perbaikan Logika Pemotongan)
    # Mengambil teks di antara 'menimbang' dan 'mengadili' secara lebih luas
    fakta_start = raw_text.find("menimbang")
    amar_start = raw_text.rfind("mengadili") # Gunakan rfind untuk mencari 'mengadili' terakhir/utama

    if fakta_start != -1 and amar_start != -1 and amar_start > fakta_start:
        fakta_hukum = raw_text[fakta_start:amar_start].strip()
        amar_putusan = raw_text[amar_start:].strip()
    else:
        # Fallback jika struktur tidak standar: ambil 5000 karakter setelah 'menimbang'
        fakta_hukum = raw_text[fakta_start:fakta_start+5000] if fakta_start != -1 else raw_text[:5000]
        amar_putusan = raw_text[amar_start:] if amar_start != -1 else ""

    return {
        "nomor_putusan": nomor_putusan,
        "tanggal": tanggal,
        "pihak": pihak,
        "pasal": pasal,
        "fakta_hukum": fakta_hukum,
        "amar_putusan": amar_putusan,
        "kategori": "Pidana"
    }

def update_case_base_v2(source_folder, output_file):
    case_base = []
    files_found = glob.glob(os.path.join(source_folder, "case_*.txt"))
    for filepath in files_found:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
            case_rep = represent_case_v2(text)
            case_base.append(case_rep)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(case_base, f, indent=4)
    print(f"[BERHASIL] Case Base diperbarui. Kasus 1 memiliki {len(case_base[0]['fakta_hukum'])} karakter fakta.")

update_case_base_v2(DATA_CLEANED, structured_file)

[BERHASIL] Case Base diperbarui. Kasus 1 memiliki 75254 karakter fakta.


In [ ]:
import pandas as pd
import os

# Memastikan direktori processed tersedia
PROCESSED_DIR = 'data/processed/'
if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)
    print(f"Direktori created: {PROCESSED_DIR}")

def export_structured_data(case_base):
    """
    Menyimpan data case base ke format JSON dan CSV
    """
    # 1. Simpan ke JSON
    json_path = os.path.join(PROCESSED_DIR, 'cases.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(case_base, f, indent=4)

    # 2. Simpan ke CSV
    csv_path = os.path.join(PROCESSED_DIR, 'cases.csv')
    df = pd.DataFrame(case_base)

    # Menyesuaikan urutan kolom sesuai permintaan user
    cols = ['nomor_putusan', 'tanggal', 'pihak', 'pasal', 'ringkasan_fakta', 'word_count', 'kategori']
    # Pastikan kolom ada sebelum reorder
    df_csv = df[[c for c in cols if c in df.columns]]
    df_csv.to_csv(csv_path, index=False, sep=';')

    print(f"[BERHASIL] Data diekspor ke:\n- {json_path}\n- {csv_path}")
    return df_csv

Direktori created: data/processed/


In [ ]:
import glob
import os
import pandas as pd

# Memastikan fungsi represent_case_v3 didefinisikan dengan benar
def represent_case_v3(raw_text):
    # Mengambil data dasar dari versi sebelumnya
    base_data = represent_case_v2(raw_text)
    words = base_data['fakta_hukum'].split()
    word_count = len(words)
    ringkasan = base_data['fakta_hukum'][:200] + "..." if len(base_data['fakta_hukum']) > 200 else base_data['fakta_hukum']

    base_data.update({
        "word_count": word_count,
        "ringkasan_fakta": ringkasan
    })
    return base_data

def process_to_final_storage(source_folder):
    case_base = []
    files_found = glob.glob(os.path.join(source_folder, "case_*.txt"))
    for filepath in sorted(files_found):
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
            case_rep = represent_case_v3(text)
            case_base.append(case_rep)

    # Memanggil fungsi export yang didefinisikan di sel f74fa13d
    return export_structured_data(case_base)

# Jalankan eksekusi
if 'DATA_CLEANED' in globals() and 'export_structured_data' in globals():
    df_final = process_to_final_storage(DATA_CLEANED)
    display(df_final.head())
else:
    print("Pastikan sel f74fa13d (export_structured_data) sudah dijalankan terlebih dahulu.")

[BERHASIL] Data diekspor ke:
- data/processed/cases.json
- data/processed/cases.csv


,nomor_putusan,tanggal,pihak,pasal,ringkasan_fakta,word_count,kategori
0,336/pid.sus/2025/pdn yyk n a i h k disclaimer ...,26 september 2025,hidayat dwi sulistiyo bin supriyaini alm h2. t...,pasal 127 ayat 1 huruf a undang undang a ri no...,"menimbang, bahw a para terdakwa diajukan ke pe...",24877,Pidana
1,418/pid.sus/2025/pdn yyk a n i h k disclaimer ...,20 oktober 2025,muhammad galang wibowo bin istanta h k 2.,pasal 436 ayat 2 undang undang republik indone...,"menimbang, b ahwa terdakwa diajukan ke persida...",8570,Pidana
2,303/pid.sus/2025/pn yyk n a i h k disclaimer k...,3 agustus 2025,sonny williams jonathan dawir alias williams a...,pasal 132d ayat 1 g uu nomor 35 tahun 2009 ten...,"menimbang, bahwa terdakwa diajukan ke persidan...",18592,Pidana
3,381/pid.sus/2025/pn yyk n a i h k disclaimer k...,2 oktober 2025,mujais bin suharib 2.,pasal 112 ayat n2 no,"menimbang, bahwa terdakwa diajukan ke persidan...",911,Pidana
4,346/pid.sus/2025/pn yyk d g n a i h k disclaim...,12 agustus 2025,n a nama lengkap arief faizal ahmad als. krite...,pasal 1l45 ayat 1 yang terkait m b dengan sedi...,"menimbang, bahwa terdakwa diajukan ke persidan...",3326,Pidana


### Tahap 3: Case Retrieval (Pencarian Kasus Serupa)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json
import os

# Daftar stop-words Indonesia yang lebih selektif
id_stop_words = [
    'yang', 'untuk', 'pada', 'ke', 'para', 'namun', 'menurut', 'antara', 'dia',
    'dua', 'ia', 'seperti', 'jika', 'sehingga', 'kembali', 'dan', 'tidak', 'ini',
    'karena', 'kepada', 'oleh', 'saat', 'harus', 'sementara', 'setelah', 'belum',
    'kami', 'sekitar', 'bagi', 'serta', 'daripada', 'dengan', 'adalah', 'bahwa'
]

def load_case_base(filepath):
    if not os.path.exists(filepath):
        return []
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)

def retrieve(query, k=5):
    """
    Implementasi Tahap 3: Retrieval dengan perbaikan pada tokenization.
    """
    cases = load_case_base(structured_file)
    if not cases:
        return "Case base kosong. Harap jalankan Tahap 1 & 2 terlebih dahulu."

    # Ambil teks fakta hukum, pastikan tidak kosong
    corpus = [c['fakta_hukum'] if c['fakta_hukum'].strip() != "" else "kosong" for c in cases]

    try:
        # Gunakan token_pattern yang lebih fleksibel untuk menangkap kata-kata hukum
        vectorizer = TfidfVectorizer(
            stop_words=id_stop_words,
            token_pattern=r"(?u)\b\w\w+\b"
        )
        tfidf_matrix = vectorizer.fit_transform(corpus)
        query_vec = vectorizer.transform([query])

        # Hitung Cosine Similarity
        similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

        # Ambil index k teratas (jangan melebihi jumlah kasus yang ada)
        k_adj = min(k, len(cases))
        top_indices = similarities.argsort()[-k_adj:][::-1]

        results = []
        for idx in top_indices:
            results.append({
                "score": float(similarities[idx]),
                "case": cases[idx]
            })

        return results
    except ValueError as e:
        return f"Gagal melakukan retrieval: {str(e)}. Periksa apakah teks cleaned mengandung kata yang valid."

In [ ]:
# Uji Coba Tahap 3: Retrieval (Re-run setelah perbaikan konten fakta)
query_hukum = "Terdakwa melakukan tindak pidana penyalahgunaan narkotika golongan I bagi diri sendiri"

print(f"Mencari kasus serupa untuk query: '{query_hukum}'\n")
hasil_retrieval = retrieve(query_hukum, k=2)

if isinstance(hasil_retrieval, list):
    for i, res in enumerate(hasil_retrieval, 1):
        print(f"Hasil #{i} (Skor Kemiripan: {res['score']:.4f})")
        print(f"- Nomor Putusan: {res['case']['nomor_putusan']}")
        print(f"- Pasal: {res['case']['pasal']}")
        print(f"- Cuplikan Fakta: {res['case']['fakta_hukum'][:200]}...")
        print("-" * 30)
else:
    print(hasil_retrieval)

Mencari kasus serupa untuk query: 'Terdakwa melakukan tindak pidana penyalahgunaan narkotika golongan I bagi diri sendiri'

Hasil #1 (Skor Kemiripan: 0.2018)
- Nomor Putusan: 324/pid.sus/2025/pdn yyk n a i h k disclaimer kepaniteraan mahkamah agunag republik indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen mahkamah agung uintuk pelayanan publik, transparansi dan akuntabilitas pelaksanaan fungsi peradilan. namun dalam hal hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang klami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu. d em al a am il h k a e l p a a n n d it a e rm m aa e n n em m u a k h a k n a m ina a k h u a r g a u s n i g in .g fo o r . m id a s i t y e a lp n g 0 te 2 r 1 m 3 u 8 a 4 t p 3 a 3 d 4 a 8 s i e tu x s t. 3 in 1 i 8 a tau informasi yang seharusnya ada, namun belum tersedia, maka harap sege b ra hubungi kepaniteraan mahkamah agung ri

### Tahap 4: Case Solution Reuse & Tahap 5: Evaluation

In [ ]:
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score

# --- TAHAP 4: CASE SOLUTION REUSE ---

def predict_outcome(query, k=3):
    """
    Memprediksi kategori/solusi hukum berdasarkan kasus-kasus paling serupa.
    """
    retrieved_results = retrieve(query, k=k)

    if isinstance(retrieved_results, str) or not retrieved_results:
        return {"prediction": "Tidak ditemukan", "confidence": 0, "top_match_score": 0}

    # Mengambil kategori dari hasil retrieval
    categories = [res['case']['kategori'] for res in retrieved_results]

    # Voting sederhana
    vote_result = Counter(categories).most_common(1)
    prediction = vote_result[0][0]
    confidence = vote_result[0][1] / len(categories)

    return {
        "prediction": prediction,
        "confidence": confidence,
        "top_match_score": retrieved_results[0]['score']
    }

# --- TAHAP 5: EVALUATION ---

def run_evaluation():
    # Contoh Query Uji & Ground Truth untuk Validasi Sistem
    test_cases = [
        {"query": "penyalahgunaan narkotika golongan 1 bagi diri sendiri", "expected": "Pidana"},
        {"query": "permufakatan jahat tanpa hak memiliki narkotika", "expected": "Pidana"},
        {"query": "sengketa warisan tanah antar ahli waris", "expected": "Perdata"}
    ]

    y_true = [tc['expected'] for tc in test_cases]
    y_pred = []

    print("=== HASIL PREDIKSI CBR ===\n")
    for tc in test_cases:
        res = predict_outcome(tc['query'])
        y_pred.append(res['prediction'])
        print(f"Query: '{tc['query']}'")
        print(f"  - Prediksi: {res['prediction']} (Conf: {res['confidence']:.2f})")
        print(f"  - Skor Kemiripan Tertinggi: {res['top_match_score']:.4f}\n")

    print("=== METRIK EVALUASI ===")
    # Zero division handled because we currently only have 'Pidana' in the base
    print(classification_report(y_true, y_pred, zero_division=0))
    print(f"Overall Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")

# Jalankan evaluasi akhir
run_evaluation()

=== HASIL PREDIKSI CBR ===

Query: 'penyalahgunaan narkotika golongan 1 bagi diri sendiri'
  - Prediksi: Pidana (Conf: 1.00)
  - Skor Kemiripan Tertinggi: 0.1064

Query: 'permufakatan jahat tanpa hak memiliki narkotika'
  - Prediksi: Pidana (Conf: 1.00)
  - Skor Kemiripan Tertinggi: 0.0568

Query: 'sengketa warisan tanah antar ahli waris'
  - Prediksi: Pidana (Conf: 1.00)
  - Skor Kemiripan Tertinggi: 0.0045

=== METRIK EVALUASI ===
              precision    recall  f1-score   support

     Perdata       0.00      0.00      0.00         1
      Pidana       0.67      1.00      0.80         2

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3

Overall Accuracy: 66.67%


### Tahap 4: Case Revision (Revisi Solusi)
Dalam tahap ini, jika skor kemiripan (similarity score) di bawah ambang batas (threshold), sistem harus meminta masukan pakar atau melakukan penyesuaian manual karena hasil retrieval dianggap kurang meyakinkan.

In [ ]:
def revise_solution(retrieval_results, threshold=0.15):
    """
    Memeriksa apakah hasil retrieval layak digunakan atau perlu revisi pakar.
    """
    if not retrieval_results or retrieval_results[0]['score'] < threshold:
        print("--- PERINGATAN: Skor kemiripan rendah ---")
        return "Perlu Revisi Pakar (Kasus Baru/Unik)"

    return retrieval_results[0]['case']['kategori']

### Tahap 5: Case Retention (Penyimpanan Kasus Baru)
Setelah solusi dikonfirmasi (baik dari retrieval atau hasil revisi), kasus tersebut disimpan kembali ke dalam `case_base.json` agar sistem semakin pintar.

In [ ]:
def retain_case(new_case_data, filepath=structured_file):
    """
    Menambahkan kasus baru yang telah divalidasi ke dalam database JSON.
    """
    current_base = load_case_base(filepath)
    current_base.append(new_case_data)

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(current_base, f, indent=4)

    print(f"[RETAIN] Kasus baru berhasil disimpan. Total kasus sekarang: {len(current_base)}")

### Simulasi Siklus Lengkap (Retrieve -> Reuse -> Revise -> Retain)

In [ ]:
# 1. Input Kasus Baru
query_baru = "Pencurian kendaraan bermotor di area parkir mall pada malam hari"

# 2. Retrieve
hasil = retrieve(query_baru, k=1)

# 3. Revise & Reuse
solusi_final = revise_solution(hasil)

# 4. Retain (Jika ini kasus baru yang valid)
if solusi_final == "Perlu Revisi Pakar (Kasus Baru/Unik)":
    # Simulasi input pakar untuk kategori
    kategori_pakar = "Pidana Umum"
    data_simpan = {
        "nomor_putusan": "NEW/2025/SIMULASI",
        "fakta_hukum": query_baru,
        "kategori": kategori_pakar
    }
    retain_case(data_simpan)

print(f"Solusi Akhir: {solusi_final if solusi_final != 'Perlu Revisi Pakar (Kasus Baru/Unik)' else kategori_pakar}")

--- PERINGATAN: Skor kemiripan rendah ---
[RETAIN] Kasus baru berhasil disimpan. Total kasus sekarang: 16
Solusi Akhir: Pidana Umum
